# Chapter 11: Matrix Decompositions: Factorization and SVD

**Book:** *Linear Algebra with Applications in Machine Learning: From Intuitive Understanding to Python Coding*

---

Matrix decompositions reveal hidden structure, improve numerical stability, and enable efficient computation. This notebook covers:

1. **Matrix Norms** -- Frobenius, spectral, L1, Linf, max
2. **Orthonormality** -- orthogonal/orthonormal vectors, normalization
3. **Orthogonal Matrices** -- properties, verification
4. **QR Factorization** -- Gram-Schmidt, least squares
5. **Symmetric Matrices and Spectral Decomposition** -- $A = QDQ^T$
6. **LU Decomposition** -- forward/backward substitution
7. **Singular Value Decomposition (SVD)** -- $A = U\Sigma V^T$, low-rank approximation, pseudoinverse
8. **ML Applications** -- PCA via SVD, image compression

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.linalg import lu, null_space

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
np.set_printoptions(precision=4, suppress=True)

print("All imports successful.")

## 11.1 Matrix Norms

Matrix norms measure the "size" of a matrix. Different norms capture different aspects of the transformation.

In [ ]:
A = np.array([[1, -2],
              [3,  4]])

# Frobenius norm: sqrt(sum of squares of all entries)
frob = np.linalg.norm(A, 'fro')

# L1 entry norm: sum of absolute entries
l1_entry = np.sum(np.abs(A))

# Linf (max row sum)
linf = np.max(np.sum(np.abs(A), axis=1))

# Max norm: largest absolute entry
max_norm = np.max(np.abs(A))

# Spectral norm: largest singular value = sqrt(max eigenvalue of A^T A)
spectral = np.linalg.norm(A, 2)

# L_{2,1} norm: sum of column L2 norms
l21 = sum(np.linalg.norm(A[:, j]) for j in range(A.shape[1]))

print(f"A =\n{A}\n")
print(f"{'Frobenius':15s} ||A||_F   = {frob:.4f}")
print(f"{'L1 (entry)':15s} sum|a_ij| = {l1_entry:.4f}")
print(f"{'L-inf':15s} max row   = {linf:.4f}")
print(f"{'Max':15s} max|a_ij| = {max_norm:.4f}")
print(f"{'Spectral':15s} sigma_max = {spectral:.4f}")
print(f"{'L_{2,1}':15s} col norms = {l21:.4f}")

In [ ]:
# Spectral norm = largest singular value
svd_vals = np.linalg.svd(A, compute_uv=False)
print(f"Singular values: {svd_vals}")
print(f"sigma_max = {svd_vals[0]:.4f} = spectral norm = {np.linalg.norm(A, 2):.4f}")
print(f"Frobenius^2 = {frob**2:.4f} = sum(sigma_i^2) = {np.sum(svd_vals**2):.4f}")

## 11.2 Orthonormality

Vectors are **orthonormal** if they are mutually orthogonal and each has unit norm. An orthonormal set forms an ideal basis: no distortion, easy projections.

In [ ]:
u = np.array([4, 2, -1])
v = np.array([1, -3, -2])

print(f"u = {u}, v = {v}")
print(f"u . v = {np.dot(u, v)}  ({'orthogonal' if np.dot(u, v) == 0 else 'not orthogonal'})")
print(f"||u|| = {np.linalg.norm(u):.4f}, ||v|| = {np.linalg.norm(v):.4f}")

# Normalize to get orthonormal pair
u_hat = u / np.linalg.norm(u)
v_hat = v / np.linalg.norm(v)
print(f"\nNormalized: u_hat = {u_hat.round(4)}, v_hat = {v_hat.round(4)}")
print(f"||u_hat|| = {np.linalg.norm(u_hat):.4f}, ||v_hat|| = {np.linalg.norm(v_hat):.4f}")
print(f"u_hat . v_hat = {np.dot(u_hat, v_hat):.6f}  (orthonormal!)")

## 11.3 Orthogonal Matrices

A square matrix $Q$ is **orthogonal** if $Q^TQ = QQ^T = I$. This means $Q^{-1} = Q^T$, and $Q$ preserves lengths and angles.

In [ ]:
s2 = np.sqrt(2) / 2
Q = np.array([[s2, -s2],
              [s2,  s2]])

print(f"Q =\n{Q.round(4)}")
print(f"Q^T Q =\n{(Q.T @ Q).round(10)}")
print(f"Orthogonal? {np.allclose(Q.T @ Q, np.eye(2))}")
print(f"det(Q) = {np.linalg.det(Q):.4f}  (1 = rotation, -1 = reflection)")

# Norm preservation
x = np.array([3, 1])
print(f"\n||x|| = {np.linalg.norm(x):.4f}")
print(f"||Qx|| = {np.linalg.norm(Q @ x):.4f}  (preserved!)")

# Non-orthogonal example
B = np.array([[0.7, -0.3], [0.3, 0.7]])
print(f"\nB =\n{B}")
print(f"B^T B =\n{(B.T @ B).round(4)}")
print(f"Orthogonal? {np.allclose(B.T @ B, np.eye(2))}  (not identity => not orthogonal)")

## 11.4 QR Factorization

$A = QR$ where $Q$ has orthonormal columns and $R$ is upper triangular. Computed via the **Gram-Schmidt process**.

In [ ]:
def gram_schmidt_qr(A):
    """QR factorization via Gram-Schmidt."""
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    
    for j in range(n):
        v = A[:, j].astype(float).copy()
        for i in range(j):
            R[i, j] = np.dot(Q[:, i], A[:, j])
            v -= R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)
        Q[:, j] = v / R[j, j]
    return Q, R

A = np.array([[1, 1, 0],
              [1, 0, 1],
              [0, 1, 1]], dtype=float)

Q_gs, R_gs = gram_schmidt_qr(A)
Q_np, R_np = np.linalg.qr(A)

print("Gram-Schmidt Q:\n", Q_gs.round(4))
print("\nGram-Schmidt R:\n", R_gs.round(4))
print(f"\nQ^T Q = I? {np.allclose(Q_gs.T @ Q_gs, np.eye(3))}")
print(f"QR = A? {np.allclose(Q_gs @ R_gs, A)}")

In [ ]:
# QR for least squares: Ax = b => Rx = Q^T b
A_ls = np.array([[1, 1],
                 [1, 2],
                 [1, 3]], dtype=float)
b_ls = np.array([1, 0, 1], dtype=float)

Q, R = np.linalg.qr(A_ls)
x_qr = np.linalg.solve(R[:2, :], (Q.T @ b_ls)[:2])
x_normal = np.linalg.inv(A_ls.T @ A_ls) @ A_ls.T @ b_ls

print(f"Least squares via QR:     x = {x_qr.round(4)}")
print(f"Least squares via normal: x = {x_normal.round(4)}")
print(f"Match: {np.allclose(x_qr, x_normal)}")

## 11.5 Symmetric Matrices and Spectral Decomposition

A symmetric matrix $A = A^T$ has real eigenvalues and orthonormal eigenvectors. The **Spectral Theorem** gives:

$$A = QDQ^T = \sum_{i=1}^n \lambda_i \mathbf{u}_i \mathbf{u}_i^T$$

In [ ]:
A_sym = np.array([[2, -2],
                  [-2, 5]], dtype=float)

eigvals, P = np.linalg.eigh(A_sym)  # eigh guarantees orthonormal eigenvectors
D = np.diag(eigvals)

print(f"A (symmetric) =\n{A_sym}")
print(f"\nEigenvalues: {eigvals}")
print(f"Eigenvectors P =\n{P.round(4)}")
print(f"\nP^T P = I? {np.allclose(P.T @ P, np.eye(2))}  (orthonormal)")
print(f"P D P^T = A? {np.allclose(P @ D @ P.T, A_sym)}")

# Rank-1 decomposition
print(f"\nSpectral decomposition as sum of rank-1 matrices:")
A_check = np.zeros_like(A_sym)
for i in range(len(eigvals)):
    Pi = np.outer(P[:, i], P[:, i])
    A_check += eigvals[i] * Pi
    print(f"  lambda_{i+1}={eigvals[i]:.2f} * u_{i+1} u_{i+1}^T =\n    {(eigvals[i] * Pi).round(4)}")
print(f"\nSum = \n{A_check.round(4)}")
print(f"Matches A? {np.allclose(A_check, A_sym)}")

In [ ]:
# 3x3 spectral decomposition
A3 = np.array([[1, 5, -2],
               [5, 4, 5],
               [-2, 5, 1]], dtype=float)

eigvals3, P3 = np.linalg.eigh(A3)
D3 = np.diag(eigvals3)

print(f"A =\n{A3}")
print(f"Eigenvalues: {eigvals3.round(4)}")
print(f"P^T P = I? {np.allclose(P3.T @ P3, np.eye(3))}")
print(f"P D P^T = A? {np.allclose(P3 @ D3 @ P3.T, A3)}")

## 11.6 LU Decomposition

$A = LU$ (or $PA = LU$ with pivoting) where $L$ is lower triangular (unit diagonal) and $U$ is upper triangular. Used for efficiently solving $A\mathbf{x} = \mathbf{b}$ via forward/backward substitution.

In [ ]:
A_lu = np.array([[2, 2, 3],
                 [5, 9, 10],
                 [4, 1, 2]], dtype=float)

P_perm, L, U = lu(A_lu)

print(f"A =\n{A_lu}")
print(f"\nPermutation P =\n{P_perm.round(4)}")
print(f"\nL (lower triangular) =\n{L.round(4)}")
print(f"\nU (upper triangular) =\n{U.round(4)}")
print(f"\nP @ L @ U = A? {np.allclose(P_perm @ L @ U, A_lu)}")

In [ ]:
# Solving Ax = b using LU
b = np.array([1, 2, 3], dtype=float)

# Forward substitution: Ly = P^T b
from scipy.linalg import solve_triangular
y = solve_triangular(L, P_perm.T @ b, lower=True)
# Backward substitution: Ux = y
x_lu = solve_triangular(U, y, lower=False)
x_direct = np.linalg.solve(A_lu, b)

print(f"Solution via LU:     x = {x_lu.round(4)}")
print(f"Solution via solve:  x = {x_direct.round(4)}")
print(f"Match: {np.allclose(x_lu, x_direct)}")

## 11.7 Singular Value Decomposition (SVD)

Any $m \times n$ matrix $A$ can be decomposed as:

$$A = U \Sigma V^T$$

- $U \in \mathbb{R}^{m \times m}$: orthogonal (left singular vectors)
- $\Sigma \in \mathbb{R}^{m \times n}$: diagonal with singular values $\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$
- $V \in \mathbb{R}^{n \times n}$: orthogonal (right singular vectors)

In [ ]:
A = np.array([[5, 5],
              [-1, 7]], dtype=float)

U, S, Vt = np.linalg.svd(A)

# Reconstruct Sigma as full matrix
Sigma = np.zeros_like(A)
np.fill_diagonal(Sigma, S)

print(f"A =\n{A}")
print(f"\nU =\n{U.round(4)}")
print(f"\nSingular values: {S.round(4)}")
print(f"\nV^T =\n{Vt.round(4)}")
print(f"\nU @ Sigma @ V^T =\n{(U @ Sigma @ Vt).round(4)}")
print(f"Matches A? {np.allclose(U @ Sigma @ Vt, A)}")

# Verify orthogonality
print(f"\nU^T U = I? {np.allclose(U.T @ U, np.eye(2))}")
print(f"V^T V = I? {np.allclose(Vt @ Vt.T, np.eye(2))}")

In [ ]:
# Non-square matrix SVD
A_rect = np.array([[0, 1, 2],
                   [1, 0, 1]], dtype=float)

U, S, Vt = np.linalg.svd(A_rect)
print(f"A ({A_rect.shape[0]}x{A_rect.shape[1]}):\n{A_rect}")
print(f"\nSingular values: {S.round(4)}")
print(f"Rank = {np.sum(S > 1e-10)}")
print(f"Spectral norm = sigma_1 = {S[0]:.4f}")
print(f"Frobenius norm = sqrt(sum sigma^2) = {np.sqrt(np.sum(S**2)):.4f}")
print(f"             np.linalg.norm(A, 'fro') = {np.linalg.norm(A_rect, 'fro'):.4f}")

### SVD Properties Summary

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 0]], dtype=float)

U, S, Vt = np.linalg.svd(A)
rank = np.sum(S > 1e-10)

print(f"A ({A.shape[0]}x{A.shape[1]}):")
print(f"  Singular values: {S.round(4)}")
print(f"  Rank: {rank}")
print(f"  Spectral norm ||A||_2 = {S[0]:.4f}")
print(f"  Frobenius norm ||A||_F = {np.sqrt(np.sum(S**2)):.4f}")
print(f"  Column space: first {rank} columns of U")
print(f"  Null space: last {A.shape[1]-rank} columns of V (= rows of V^T)")

### Visualizing SVD: Unit Circle to Ellipse

In [ ]:
A = np.array([[5, 5], [-1, 7]], dtype=float)
U, S, Vt = np.linalg.svd(A)

theta = np.linspace(0, 2*np.pi, 200)
circle = np.array([np.cos(theta), np.sin(theta)])
ellipse = A @ circle

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Input space with right singular vectors
ax1.plot(circle[0], circle[1], 'b-', linewidth=1.5)
for i in range(2):
    v = Vt[i]
    ax1.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1,
              color=['red', 'orange'][i], linewidth=2.5, label=f'v{i+1}')
ax1.set_title('Input Space: Unit Circle + Right Singular Vectors', fontsize=12)
ax1.set_xlim(-1.5, 1.5)
ax1.set_ylim(-1.5, 1.5)
ax1.set_aspect('equal')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Output space: ellipse with left singular vectors scaled by sigma
ax2.plot(ellipse[0], ellipse[1], 'b-', linewidth=1.5)
for i in range(2):
    u = U[:, i] * S[i]
    ax2.quiver(0, 0, u[0], u[1], angles='xy', scale_units='xy', scale=1,
              color=['red', 'orange'][i], linewidth=2.5,
              label=f'sigma{i+1}*u{i+1} ({S[i]:.1f})')
ax2.set_title('Output Space: Ellipse + Scaled Left Singular Vectors', fontsize=12)
lim = np.max(np.abs(ellipse)) + 1
ax2.set_xlim(-lim, lim)
ax2.set_ylim(-lim, lim)
ax2.set_aspect('equal')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('SVD: Circle -> Ellipse', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Low-Rank Approximation

The best rank-$k$ approximation is $A_k = \sum_{i=1}^k \sigma_i \mathbf{u}_i \mathbf{v}_i^T$, minimizing $\|A - A_k\|_F$.

In [ ]:
np.random.seed(42)
A_big = np.random.randn(50, 50)
# Make it have a fast-decaying spectrum
U_r, _, Vt_r = np.linalg.svd(A_big)
S_decay = np.exp(-0.3 * np.arange(50))
A_big = U_r @ np.diag(S_decay) @ Vt_r

U, S, Vt = np.linalg.svd(A_big)

# Compute error for different ranks
ranks = range(1, 20)
errors = []
variances = []
total_energy = np.sum(S**2)

for k in ranks:
    A_k = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    err = np.linalg.norm(A_big - A_k, 'fro') / np.linalg.norm(A_big, 'fro')
    var = np.sum(S[:k]**2) / total_energy
    errors.append(err)
    variances.append(var)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ranks, errors, 'bo-', markersize=5)
ax1.set_xlabel('Rank k')
ax1.set_ylabel('Relative Frobenius Error')
ax1.set_title('Low-Rank Approximation Error', fontsize=12)
ax1.grid(True, alpha=0.3)

ax2.plot(ranks, [v * 100 for v in variances], 'ro-', markersize=5)
ax2.set_xlabel('Rank k')
ax2.set_ylabel('Variance Captured (%)')
ax2.set_title('Energy / Variance Retained', fontsize=12)
ax2.axhline(95, color='gray', linestyle='--', label='95%')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('SVD Low-Rank Approximation Quality', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Solving Linear Systems with SVD (Pseudoinverse)

For any system $A\mathbf{x} = \mathbf{b}$, the SVD gives the pseudoinverse solution $\mathbf{x} = A^+ \mathbf{b} = V \Sigma^+ U^T \mathbf{b}$.

In [ ]:
# Overdetermined system (m > n): least squares
A_od = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
b_od = np.array([1, 0, 1], dtype=float)

x_pinv = np.linalg.pinv(A_od) @ b_od
x_lstsq = np.linalg.lstsq(A_od, b_od, rcond=None)[0]

print("=== Overdetermined (3 equations, 2 unknowns) ===")
print(f"x (pseudoinverse) = {x_pinv.round(4)}")
print(f"x (lstsq)         = {x_lstsq.round(4)}")
print(f"Residual: b - Ax = {(b_od - A_od @ x_pinv).round(4)}")
print(f"Residual norm = {np.linalg.norm(b_od - A_od @ x_pinv):.4f}")

# Underdetermined system (m < n): minimum-norm solution
A_ud = np.array([[1, 1, 1]], dtype=float)
b_ud = np.array([3], dtype=float)

x_ud = np.linalg.pinv(A_ud) @ b_ud
print(f"\n=== Underdetermined (1 equation, 3 unknowns) ===")
print(f"x (minimum norm) = {x_ud.round(4)}")
print(f"Ax = {(A_ud @ x_ud).round(4)}, ||x|| = {np.linalg.norm(x_ud):.4f}")

# Rank-deficient system
A_rd = np.array([[1, 1], [2, 2], [3, 3]], dtype=float)
b_rd = np.array([1, 2, 3], dtype=float)

x_rd = np.linalg.pinv(A_rd) @ b_rd
print(f"\n=== Rank-deficient (rank 1) ===")
print(f"x (pseudoinverse) = {x_rd.round(4)}")
print(f"Residual norm = {np.linalg.norm(b_rd - A_rd @ x_rd):.4f}")

## 11.8 ML Applications

### PCA via SVD

In [ ]:
np.random.seed(42)
data = np.random.multivariate_normal([0, 0], [[3, 2], [2, 2]], 200)
X_centered = data - data.mean(axis=0)

# PCA via SVD of centered data
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)

# Singular values relate to variance: var_i = sigma_i^2 / (n-1)
explained_var = S**2 / (len(data) - 1)
explained_ratio = explained_var / explained_var.sum() * 100

print(f"Singular values: {S[:2].round(4)}")
print(f"Explained variance ratio: {explained_ratio.round(1)}%")
print(f"Principal directions (rows of V^T):\n{Vt.round(4)}")

# Project to 1D
X_1d = X_centered @ Vt[0]  # project onto PC1
X_reconstructed = np.outer(X_1d, Vt[0])  # reconstruct in 2D

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.3, s=15, label='Data')
ax.scatter(X_reconstructed[:, 0], X_reconstructed[:, 1], alpha=0.3, s=15,
           color='red', label='Projected (1D)')

for i, c in enumerate(['red', 'orange']):
    direction = Vt[i] * S[i] / np.sqrt(len(data) - 1) * 3
    ax.annotate('', xy=direction, xytext=-direction,
                arrowprops=dict(arrowstyle='<->', color=c, lw=3))
    ax.text(*(direction * 1.15), f'PC{i+1} ({explained_ratio[i]:.0f}%)',
            fontsize=11, fontweight='bold', color=c)

ax.set_aspect('equal')
ax.set_title('PCA via SVD', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Image Compression via Truncated SVD

In [ ]:
# Create a synthetic grayscale image
np.random.seed(7)
x_grid, y_grid = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
img = np.sin(x_grid) * np.cos(y_grid) + 0.3 * np.random.randn(200, 200)

U_img, S_img, Vt_img = np.linalg.svd(img, full_matrices=False)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
ks = [1, 5, 20, 200]
for ax, k in zip(axes, ks):
    img_k = U_img[:, :k] @ np.diag(S_img[:k]) @ Vt_img[:k, :]
    err = np.linalg.norm(img - img_k, 'fro') / np.linalg.norm(img, 'fro')
    ax.imshow(img_k, cmap='gray')
    ax.set_title(f'Rank {k}\nError: {err:.2%}', fontsize=11)
    ax.axis('off')

plt.suptitle('SVD Image Compression at Various Ranks', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 11.9 Exercises

Selected exercises from the chapter.

**Exercise 1:** Compute all norms for $A = \begin{bmatrix} 2 & -1 \\ 0 & 3 \end{bmatrix}$.

In [ ]:
# Exercise 1: Your code here


**Exercise 5:** Compute QR factorization of $A = \begin{bmatrix} 1 & 0 \\ 1 & 1 \\ 1 & -1 \end{bmatrix}$.

In [ ]:
# Exercise 5: Your code here


**Exercise 8:** Derive spectral decomposition of $A = \begin{bmatrix} 1 & 2 \\ 2 & 1 \end{bmatrix}$.

In [ ]:
# Exercise 8: Your code here


**Exercise 14:** Compute the singular values of $A = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}$.

In [ ]:
# Exercise 14: Your code here


---

## Exercise Solutions

In [ ]:
# --- Solution: Exercise 1 ---
A = np.array([[2, -1], [0, 3]])
print(f"Frobenius: {np.linalg.norm(A, 'fro'):.4f}")
print(f"Spectral:  {np.linalg.norm(A, 2):.4f}")
print(f"L-inf:     {np.max(np.sum(np.abs(A), axis=1)):.4f}")
print(f"Max:       {np.max(np.abs(A)):.4f}")
print(f"L1 entry:  {np.sum(np.abs(A)):.4f}")

In [ ]:
# --- Solution: Exercise 5 ---
A = np.array([[1, 0], [1, 1], [1, -1]], dtype=float)
Q, R = np.linalg.qr(A)
print(f"Q =\n{Q.round(4)}")
print(f"R =\n{R.round(4)}")
print(f"Q^T Q = I? {np.allclose(Q.T @ Q, np.eye(2))}")
print(f"QR = A? {np.allclose(Q @ R, A)}")

In [ ]:
# --- Solution: Exercise 8 ---
A = np.array([[1, 2], [2, 1]], dtype=float)
eigvals, P = np.linalg.eigh(A)
print(f"Eigenvalues: {eigvals}")
print(f"P =\n{P.round(4)}")
print(f"A = P D P^T? {np.allclose(P @ np.diag(eigvals) @ P.T, A)}")
for i in range(2):
    print(f"  {eigvals[i]:.0f} * u{i+1} u{i+1}^T =\n    {(eigvals[i] * np.outer(P[:, i], P[:, i])).round(4)}")

In [ ]:
# --- Solution: Exercise 14 ---
A = np.array([[2, 1], [1, 2]], dtype=float)
S = np.linalg.svd(A, compute_uv=False)
print(f"Singular values: {S.round(4)}")
print(f"These are sqrt of eigenvalues of A^T A:")
eigvals_ATA = np.linalg.eigvalsh(A.T @ A)
print(f"  eigenvalues of A^T A: {np.sort(eigvals_ATA)[::-1].round(4)}")
print(f"  sqrt: {np.sqrt(np.sort(eigvals_ATA)[::-1]).round(4)}")